In [0]:
# Imports
import requests
import time
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType
)
from pyspark.sql.functions import lit, current_timestamp

print("Imports successful")

In [0]:
# Countries with coordinates
COUNTRIES = [
    {"country_code": "KE", "country_name": "Kenya",        "lat": -1.286,  "lon": 36.817},
    {"country_code": "UG", "country_name": "Uganda",       "lat":  0.347,  "lon": 32.583},
    {"country_code": "TZ", "country_name": "Tanzania",     "lat": -6.173,  "lon": 35.739},
    {"country_code": "ET", "country_name": "Ethiopia",     "lat":  9.025,  "lon": 38.747},
    {"country_code": "GH", "country_name": "Ghana",        "lat":  5.603,  "lon": -0.187},
    {"country_code": "NG", "country_name": "Nigeria",      "lat":  9.058,  "lon":  7.499},
    {"country_code": "ZA", "country_name": "South Africa", "lat": -25.746, "lon": 28.188},
    {"country_code": "SN", "country_name": "Senegal",      "lat": 14.693,  "lon": -17.447},
    {"country_code": "ML", "country_name": "Mali",         "lat": 12.650,  "lon": -8.000},
    {"country_code": "MZ", "country_name": "Mozambique",   "lat": -25.966, "lon": 32.573},
    {"country_code": "RW", "country_name": "Rwanda",       "lat": -1.943,  "lon": 29.874},
    {"country_code": "ZM", "country_name": "Zambia",       "lat": -15.417, "lon": 28.283},
    {"country_code": "ZW", "country_name": "Zimbabwe",     "lat": -17.829, "lon": 31.052},
    {"country_code": "CM", "country_name": "Cameroon",     "lat":  3.848,  "lon": 11.502},
    {"country_code": "CI", "country_name": "Cote d'Ivoire","lat":  5.354,  "lon": -4.005},
]
print(f"Defined {len(COUNTRIES)} countries")

In [0]:
# Fetch function with retry + backoff built in from the start
def fetch_climate(country: dict, start: str, end: str) -> list:
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":   country["lat"],
        "longitude":  country["lon"],
        "start_date": start,
        "end_date":   end,
        "daily":      "precipitation_sum,temperature_2m_mean",
        "timezone":   "UTC",
    }
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    dates  = data["daily"]["time"]
    precip = data["daily"]["precipitation_sum"]
    temps  = data["daily"]["temperature_2m_mean"]

    records = []
    for date, p, t in zip(dates, precip, temps):
        records.append({
            "country_code": country["country_code"],
            "country_name": country["country_name"],
            "latitude":     country["lat"],
            "longitude":    country["lon"],
            "date":         date,
            "year":         int(date[:4]),
            "month":        int(date[5:7]),
            "precipitation_mm":   float(p) if p is not None else None,
            "temperature_c_mean": float(t) if t is not None else None,
        })
    return records


def fetch_climate_with_retry(country: dict, start: str, end: str, max_retries: int = 4) -> list:
    wait_time = 10
    for attempt in range(max_retries + 1):
        try:
            return fetch_climate(country, start, end)
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 429 and attempt < max_retries:
                print(f"  Rate limited. Waiting {wait_time}s, retry {attempt + 1}/{max_retries}...")
                time.sleep(wait_time)
                wait_time *= 2
            else:
                raise

print("Fetch functions ready")

In [0]:
# Fetch ALL 15 countries in one clean pass

all_records = []

for country in COUNTRIES:
    print(f"Fetching: {country['country_name']}...", end=" ")
    try:
        records = fetch_climate_with_retry(country, "2010-01-01", "2023-12-31")
        all_records.extend(records)
        print(f"{len(records)} daily records")
    except Exception as e:
        print(f"FAILED — {e}")
    time.sleep(5)

print(f"\nTotal records fetched: {len(all_records)}")
print(f"Expected: {15 * 5113} (15 countries × 5113 days)")

In [0]:
# Cell 5 — schema and DataFrame
schema = StructType([
    StructField("country_code",       StringType(),  True),
    StructField("country_name",       StringType(),  True),
    StructField("latitude",           DoubleType(),  True),
    StructField("longitude",          DoubleType(),  True),
    StructField("date",               StringType(),  True),
    StructField("year",               IntegerType(), True),
    StructField("month",              IntegerType(), True),
    StructField("precipitation_mm",   DoubleType(),  True),
    StructField("temperature_c_mean", DoubleType(),  True),
])

df_climate = spark.createDataFrame(all_records, schema=schema)

df_climate = (df_climate
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source",      lit("open_meteo_api"))
    .withColumn("indicator",   lit("climate_daily"))
)

print(f"DataFrame: {df_climate.count()} rows, {len(df_climate.columns)} columns")

In [0]:
# Write to Unity Catalog
(df_climate.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.bronze_open_meteo_climate")
)

print("Written to Unity Catalog: workspace.default.bronze_open_meteo_climate")

In [0]:
# Cell 7 — verify
df_check = spark.table("workspace.default.bronze_open_meteo_climate")
print(f"Total rows: {df_check.count()}")
display(df_check.groupBy("country_code", "country_name").count().orderBy("country_code"))

In [0]:
# Final Bronze audit — confirm all five tables are intact
tables = [
    "bronze_world_bank_gdp",
    "bronze_open_meteo_climate",
    "bronze_world_bank_food",
    "bronze_world_bank_population",
    "dim_country",
]

for table in tables:
    try:
        count = spark.table(f"workspace.default.{table}").count()
        print(f"{table}: {count} rows")
    except Exception as e:
        print(f"{table}: FAILED — {e}")